### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_interface_audit_appointment
- _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item
- _exponent._bronze_allscripts_tw_works.dbo_item_medication
- _exponent._bronze_allscripts_tw_works.dbo_person_other (for DateOfDeath capping)

### Strategy:
- Aggregate MIN/MAX clinical activity dates per patient across all source tables
- Create one observation period per patient (no gap splitting)
- Cap end date at DateOfDeath if patient is deceased
- Use period_type_concept_id = 32817 (EHR encounter record)

### Notes:
- This notebook depends on source_to_person being populated
- Observation period represents the span of time during which the patient has data in the EHR

In [0]:
source = 'allscripts_tw'

# Transformation

In [0]:
silver_observation_period_df = spark.sql(f'''
WITH clinical_activity AS (
  -- Appointments
  SELECT 
    CAST(Patientid AS BIGINT) AS PatientID, 
    EncounterDTTM AS activity_date 
  FROM _exponent._bronze_allscripts_tw_works.dbo_interface_audit_appointment
  WHERE EncounterDTTM IS NOT NULL
    AND Patientid IS NOT NULL
  
  UNION ALL
  
  -- Order Activity
  SELECT 
    CAST(PatientID AS BIGINT) AS PatientID, 
    CreateDTTM AS activity_date 
  FROM _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
  WHERE CreateDTTM IS NOT NULL
    AND PatientID IS NOT NULL
  
  UNION ALL
  
  -- Vendor Items (conditions, procedures, observations)
  SELECT 
    CAST(PatientID AS BIGINT) AS PatientID, 
    COALESCE(PerformedDTTM, RecordedDTTM, CreateDTTM) AS activity_date
  FROM _exponent._bronze_allscripts_tw_works.dbo_vendor_item
  WHERE COALESCE(PerformedDTTM, RecordedDTTM, CreateDTTM) IS NOT NULL
    AND PatientID IS NOT NULL
    AND IsErrorFLAG = 'N'
  
  UNION ALL
  
  -- Medications
  SELECT 
    CAST(PatientID AS BIGINT) AS PatientID, 
    COALESCE(PerformedDTTM, CreateDTTM, LastUpdateDTTM) AS activity_date
  FROM _exponent._bronze_allscripts_tw_works.dbo_item_medication
  WHERE COALESCE(PerformedDTTM, CreateDTTM, LastUpdateDTTM) IS NOT NULL
    AND PatientID IS NOT NULL
    AND IsErrorFLAG = 'N'
),

-- Aggregate per patient: MIN start, MAX end
patient_observation_window AS (
  SELECT 
    PatientID,
    MIN(DATE(activity_date)) AS observation_start,
    MAX(DATE(activity_date)) AS observation_end
  FROM clinical_activity
  WHERE activity_date >= '1900-01-01'  -- Filter invalid dates
    AND activity_date <= CURRENT_DATE()  -- Filter future dates
  GROUP BY PatientID
)

-- Final select with death date capping
SELECT
  pow.observation_start AS observation_period_start_date,
  CASE 
    WHEN po.DateOfDeath IS NOT NULL AND DATE(po.DateOfDeath) < pow.observation_end 
    THEN DATE(po.DateOfDeath)
    ELSE pow.observation_end 
  END AS observation_period_end_date,
  32817 AS period_type_concept_id,  -- EHR encounter record
  CONCAT('{source}', ' | ', CAST(pow.PatientID AS STRING)) AS person_source_value,
  CONCAT('{source}', ' | ', CAST(pow.PatientID AS STRING)) AS observation_period_source_value,
  '{source}' AS source_system
FROM patient_observation_window pow
LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_person_other po
  ON pow.PatientID = CAST(po.ID AS BIGINT)
-- Only include patients that exist in source_to_person mapping
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(pow.PatientID AS STRING))
  AND stp.active_flag = TRUE
''')

display(silver_observation_period_df)
silver_observation_period_df.createOrReplaceTempView("silver_observation_period")

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.observation_period AS t
USING silver_observation_period AS s
ON t.observation_period_source_value = s.observation_period_source_value

WHEN MATCHED AND (
     NOT (t.observation_period_start_date <=> s.observation_period_start_date)
  OR NOT (t.observation_period_end_date <=> s.observation_period_end_date)
  OR NOT (t.period_type_concept_id <=> s.period_type_concept_id)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.observation_period_start_date = s.observation_period_start_date,
  t.observation_period_end_date   = s.observation_period_end_date,
  t.period_type_concept_id        = s.period_type_concept_id,
  t.person_source_value           = s.person_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id,
  person_source_value,
  observation_period_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.observation_period_start_date,
  s.observation_period_end_date,
  s.period_type_concept_id,
  s.person_source_value,
  s.observation_period_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
%sql
-- Verify silver layer
SELECT * FROM _exponent.omop_silver.observation_period LIMIT 10

In [0]:
%sql
-- Insert new mappings to source_to_observation_period
INSERT INTO _exponent.omop_mapping.source_to_observation_period (
    source_system,
    observation_period_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.observation_period_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, observation_period_source_value, last_mod_tsp
    FROM _exponent.omop_silver.observation_period
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation_period x
  ON s.observation_period_source_value = x.observation_period_source_value;

In [0]:
%sql
-- Verify mapping table
SELECT * FROM _exponent.omop_mapping.source_to_observation_period LIMIT 10

In [0]:
%sql
-- Merge to Gold layer
MERGE INTO _exponent.omop.observation_period AS gold
USING (
  SELECT
    sop.observation_period_id,
    stp.person_id,
    s.observation_period_start_date,
    s.observation_period_end_date,
    s.period_type_concept_id
  FROM _exponent.omop_silver.observation_period s
  JOIN _exponent.omop_mapping.source_to_observation_period sop
    ON sop.observation_period_source_value = s.observation_period_source_value
   AND sop.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
) AS src
ON gold.observation_period_id = src.observation_period_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                       = src.person_id,
  gold.observation_period_start_date   = src.observation_period_start_date,
  gold.observation_period_end_date     = src.observation_period_end_date,
  gold.period_type_concept_id          = src.period_type_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_period_id,
  person_id,
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id
)
VALUES (
  src.observation_period_id,
  src.person_id,
  src.observation_period_start_date,
  src.observation_period_end_date,
  src.period_type_concept_id
);

In [0]:
%sql
-- Verify gold layer
SELECT * FROM _exponent.omop.observation_period LIMIT 10

In [0]:
%sql
-- Validation: Count records at each layer
SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.observation_period
UNION ALL
SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_observation_period
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.observation_period